In [ ]:
#import hvplot.pandas
#from bokeh.sampledata.penguins import data as df

#df.hvplot.scatter(x='bill_length_mm', y='bill_depth_mm', by='species')

In [ ]:
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
from math import cos, asin, sqrt
import re

import numpy as np
import geopandas as gpd
import pandas as pd
from matplotlib import pyplot as plt
import os
import rioxarray as rio
import xarray as xr
import rasterio
import glob
from shapely.errors import ShapelyDeprecationWarning
from shapely.geometry import Point
import warnings
import folium
import datetime
import time
from folium import plugins
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
#import contextily as cx
from shapely.geometry import box
import sys
from datetime import datetime, timedelta
from itertools import chain

from datetime import date
from bs4 import BeautifulSoup
import requests

In [ ]:
sys.path.insert(0, '/projects/old_shared/fire_weather_vis/base-fwi-vis/')
import fwiVis.fwiVis as fv

path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_only/April_1_unmerged_fires_with_FWI.csv"
#path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Final_dataset_as_of_20240209.csv"
fire3 = fv.prep_fire_files(path)

ciffc = pd.read_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/CIFFC_data/ciffc_all_canada.csv")
ciffc = ciffc[ciffc.field_agency_code == "qc"]

ciffc = gpd.GeoDataFrame(ciffc, geometry= gpd.points_from_xy(ciffc.field_longitude, ciffc.field_latitude), crs = "4326")
ciffc = ciffc.to_crs("3571")

In [ ]:
fire3 = fire3.sort_values(by = ["fireID", "t"])
fire3 = fire3[~fire3.FWI.isna()]
#fire3.farea = fire3.farea.astype("int64")
fire3 = fire3.sjoin(ciffc)
fire3["farea_diff"] = fire3.groupby("fireID").farea.diff()

row_mask = (~fire3.fireID.str.contains("_"))

#fire3[row_mask].hvplot.scatter(x='FWI', y='farea_diff', hover_cols=['fireID', 't'])
#fire3[row_mask].plot.scatter(x='FWI', y='farea_diff')

In [ ]:
fire3[fire3.field_agency_fire_id == "534"].field_response_type.unique()

In [ ]:
fire3 = fire3.sort_values(by = ["fireID", "t"])
fire3 = fire3[~fire3.FWI.isna()]
#fire3.farea = fire3.farea.astype("int64")
fire3["farea_diff"] = fire3.groupby("fireID").farea.diff()

row_mask = (~fire3.fireID.str.contains("_"))
fire3[row_mask]
#fire3[row_mask].hvplot.scatter(x='FWI', y='farea_diff', hover_cols=['fireID', 't'])
#fire3[row_mask].plot.scatter(x='FWI', y='farea_diff')

In [ ]:
fire3["FWI_diff"] = fire3.groupby("fireID").FWI.diff()
fire3["farea_diff"] = fire3.groupby("fireID").farea.diff()

row_mask = (~fire3.fireID.str.contains("_"))

#fire3[row_mask].hvplot.scatter(x='FWI_diff', y='farea_diff', hover_cols=['fireID', 't'])
#fire3[row_mask].plot.scatter(x='FWI', y='farea_diff')

In [ ]:
fire3["FWI_rolling"] = fire3.groupby("fireID").FWI.rolling(3).max().reset_index(drop = True)

#fire3.groupby("fireID").FWI.rolling(3).mean()

In [ ]:
row_mask = (~fire3.fireID.str.contains("_"))

#fire3[row_mask].hvplot.scatter(x='FWI_rolling', y='farea_diff', hover_cols=['fireID', 't'])

In [ ]:
row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1)

#fire3[row_mask].hvplot.scatter(x='FWI_rolling', y='farea', hover_cols=['fireID', 't'])

In [ ]:
# test = fire3.groupby("fireID").day_of_fire.max()
# test



### Get fireIDs that last longer than three days

def normalize_fwi(df, col = "FWI"):
    #print(df[col].mean())
    df[col+"_norm"] = df[col]/df[col].mean()
    return(df)

fire3 = fire3.groupby("fireID").apply(normalize_fwi).reset_index(drop = True)

In [ ]:
len(fire3[(fire3.field_response_type == "MON") & (~fire3.fireID.str.contains("_"))].fireID.unique())

In [ ]:
len(fire3[(~fire3.fireID.str.contains("_"))].fireID.unique())

len(fire3[(~fire3.fireID.str.contains("_"))].field_agency_fire_id.unique())

In [ ]:
ids_from_v3 = ['462', '453', '468', '469', '480', '479', '470', '473', '492',
       '483', '490', '481', '501', '525', '502', '498', '503', '500',
       '508', '504', '172', '378', '520', '521', '519', '526', '219',
       '218', '485', '516', '222', '258', '311', '266', '309', '376',
       '312', '296', '197', '237', '268', '304', '419', '292', '344',
       '420', '345', '402', '260', '278', '262', '64', '357', '559',
       '571', '565', '542', '545', '547', '544', '553', '548', '541',
       '550', '630', '549', '618', '617', '546', '551', '225', '224',
       '556', '555', '543', '540', '566', '557', '643', '552', '554',
       '564', '558', '563', '245', '562', '569', '226', '277', '283',
       '257', '270', '574', '575', '577', '579', '306', '279', '583',
       '581', '602', '587', '614', '601', '589', '588', '287', '243',
       '242', '299', '240', '284', '598', '584', '586', '590', '597',
       '595', '608', '606', '302', '603', '233', '314', '234', '235',
       '613', '604', '593', '605', '693', '619', '623', '624', '236',
       '232', '668', '622', '578', '271', '626', '281', '701', '280',
       '313', '629', '621', '341', '339', '607', '248', '250', '359',
       '694', '297', '382', '334', '368', '367', '466', '300', '379',
       '369', '274', '370', '371', '449', '351', '686', '373', '353',
       '308', '305', '295', '628', '418', '352', '670', '689', '699',
       '361', '256', '326', '375', '252', '261', '269', '263', '267',
       '385', '249', '381', '265', '411', '393', '408', '412', '450',
       '459', '454', '460', '478', '461', '451', '457']

ids_from_v3 = pd.DataFrame({"ids": ids_from_v3})
### Find IDS from v2 not in v3

miss_v3 = fire3[~fire3.field_agency_fire_id.isin(ids_from_v3.ids)].field_agency_fire_id.unique()
print(fire3[~fire3.field_agency_fire_id.isin(ids_from_v3.ids)].field_agency_fire_id.unique())


### from IDS from v3 not in v2
miss_v2 = ids_from_v3[~ids_from_v3.ids.isin(fire3.field_agency_fire_id.unique())].ids.unique()
print(ids_from_v3[~ids_from_v3.ids.isin(fire3.field_agency_fire_id.unique())].ids.unique())

In [ ]:
fire3 = fire3.groupby("fireID").apply(get_final)

In [ ]:
plot_fire = fire3[fire3.is_final]

In [ ]:
plot_fire

In [ ]:

m = plot_fire.explore()
ciffc[ciffc.field_agency_fire_id.isin(miss_v3)].explore(m = m,  color = "red")


In [ ]:
m = plot_fire.explore()
ciffc[ciffc.field_agency_fire_id.isin(miss_v2)].explore(m = m,  color = "red")

In [ ]:
ciffc[ciffc.field_agency_fire_id.isin(miss_v2)]

In [ ]:
ciffc[ciffc.field_agency_fire_id.isin(miss_v3)]

In [ ]:
#import numpy as np
from matplotlib import pyplot as plt
from plotnine import ggplot, geom_point, geom_jitter, aes, stat_smooth, facet_wrap
import plotnine as plotnine

### Some useful vars to color by 
def assign_day_of_fire(df):
    df = df.sort_values(by = "t")
    df['day_of_fire'] = df.t.rank()
    #df['day_of_fire'] = df['day_of_fire'].astype("int64")
    return(df)


fire3 = fire3.groupby("fireID").apply(assign_day_of_fire).reset_index(drop = True)
fire3["farea_shifted"] = fire3.groupby("fireID").farea.shift(periods = 1)

fire3["normalized_farea_diff"] = fire3.farea_diff/fire3.farea_shifted

rolling_num = 3
agg_function = "max" # max

fire3["FWI_rolling"] = fire3.groupby("fireID").FWI.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
#fire3["FWI_norm_rolling"] = fire3.groupby("fireID").FWI_norm.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3["FWI_diff_rolling"] = fire3.groupby("fireID").FWI_diff.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3['farea_rolling'] = fire3.groupby("fireID").farea.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3['farea_diff_rolling'] = fire3.groupby("fireID").farea_diff.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3["GEOS-5.IMERGEARLY_rolling"] = fire3.groupby("fireID")["GEOS-5.IMERGEARLY"].rolling(rolling_num).agg(agg_function).reset_index(drop = True)

long_fires = fire3[fire3.day_of_fire > 3].fireID.unique()
#fire3['max_dof'] = fire3.groupby("fireID").day_of_fire.max()

row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)

#x_var = ["FWI","FWI_rolling", "FWI_diff_rolling", "FWI_norm_rolling"]
#y_var = ['farea', "normalized_farea_diff", 'farea_rolling', 'farea_diff_rolling']


x_var = ["FWI","FWI_rolling", "FWI_diff_rolling", "GEOS-5.IMERGEARLY", "GEOS-5.IMERGEARLY_rolling"]
y_var = [ 'farea_diff_rolling']

for x in x_var:
    for y in y_var:
        #p = (ggplot(fire3[row_mask], aes( x = x, y = y, color = 'field_latitude'))
        p = (ggplot(fire3[row_mask], aes( x = x, y = y, color = "field_response_type"))
        #p = (ggplot(fire3[row_mask], aes( x = x, y = y, color = "farea_shifted"))
         + geom_point()
         + plotnine.labels.ylab(y)
         + plotnine.labels.xlab(x)
         + stat_smooth(method = "glm", formula = "y ~ x")
         #+ plotnine.scale_y_log10()
         + plotnine.ggtitle(f"{agg_function} in rolling window of {rolling_num} days")

         )
        print(p)
        #del(p)

        
        

## Trying to narrow things down to just publication-level figures. 

In [ ]:
# fire3.columns
fire3["GEOS5_IMERGEARLY_rolling"] = fire3["GEOS-5.IMERGEARLY_rolling"]

fire3["log_farea_diff_rolling"] = np.log(fire3["farea_diff_rolling"] + 0.999)

In [ ]:
#zip_model = sm.ZeroInflatedPoisson.from_formula(formula, data=df, inflation='probit').fit()
#print(zip_model.summary())

In [ ]:
# print(zip_model.aic)

# ?sm.ZeroInflatedPoisson

In [ ]:
#smf.glm
fire3.loc[row_mask, ['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling"]].field_response_type

In [ ]:
### Need to implement a color bar for fill_between
# rgb_values.reverse()
# rgb_values

#pretty_names = {c: "Response Type", "MON": "Monitored", "FUL": "Full Supression", "MOD": "Modified Supression"}

#fire3.field_response_type.map(pretty_names).unique()


In [ ]:
### Trying to fit a glm for non-constant varience 
from patsy import ModelDesc
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf


row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)
#row_mask = (~fire3.fireID.str.contains("_")) 

df = fire3[row_mask]

df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling"]].dropna()




y = "farea_diff_rolling"
x = "GEOS5_IMERGEARLY_rolling"
c = "field_response_type"

pretty_names = {c: "Response Type", "MON": "Monitored", "FUL": "Full Supression", "MOD": "Modified Supression"}

# Unique category labels: 'D', 'F', 'G', ...
color_labels = df[c].unique()

# List of RGB triplets
rgb_values = sns.color_palette("Set1", 3)
rgb_values.reverse()

# Map label to RGB
color_map = dict(zip(color_labels, rgb_values))

# # Finally use the mapped values
# plt.scatter(df['carat'], df['price'], c=df[c].map(color_map))


formula = f"{y} ~ {x}:C({c})"
print(formula)

families = ["Gaussian", "Gamma", "Poisson", "NegativeBinomial"]
links = ["Log", "Identity"]
#links = ["Identity"]

ls = []
fs = []
aic = []
ll = []
bic = []
for f in families:
    for l in links: 
        #desc = ModelDesc.from_formula(formula)
        #desc.describe()

        #sm.families.family.Gamma.links
        #link_g = sm.genmod.families.links.Identity()
        #link_g = sm.genmod.families.links.Log()
        #link_g = sm.genmod.families.links.CLogLog()
        #link_g = sm.genmod.families.links.Sqrt()
        #link_g = sm.genmod.families.links.InversePower()
        #link_g = sm.genmod.families.links.NegativeBinomial()
        #link_g = sm.genmod.families.links.Power()
        
        link_g = getattr(sm.genmod.families.links, l)

        method = getattr(sm.families, f)

        model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
        print(model.summary())
        tmp = model.summary2()
        
        fs.append(f)
        ls.append(l)
        aic.append(model.aic)
        ll.append(tmp.tables[0].iloc[2,3])
        bic.append(model.bic)



        df['fitted'] = model.fittedvalues
        df['residuals'] = model.resid_response
        df = df.sort_values(by = x)
        # Plot residuals vs fitted values
        res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
        res
        plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
        plt.xlabel('Fitted Values')
        plt.ylabel('Residuals')
        plt.show()
        
    
        predictions = model.get_prediction(df, transform = True) #df, transform = False
        df['predicted'] = predictions.predicted_mean
        df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T

        actual = sns.scatterplot(x=x, y=y, data=df, hue = c, palette = color_map) #facet_kws={'legend_out': True}
        
        handles, labels = actual.get_legend_handles_labels()

        # Customize legend titles and labels
        actual.legend(handles=handles, labels=[ "Monitored", "Full Suppression", "Modified Suppression"], title=pretty_names[c])
        # actual._legend.set_title(pretty_names[c])
        #  # replace labels
        # for t, l in zip(actual._legend.texts, df.field_response_type.map(pretty_names).unique()):
        #     t.set_text(l)
        #Plot the fitted values
        pred = sns.lineplot(x=x, y='predicted', data=df, hue =c, legend = False, palette = color_map)
        pred
        # Plot the confidence intervals
        for cat in df[c].unique():
            #print(cat)
            #print(df.loc[(df[c] == cat), [c]].map(color_map))
            plt.fill_between(df.loc[(df[c] == cat)][x], df.loc[(df[c] == cat)]['conf_int_low'], df.loc[(df[c] == cat)]['conf_int_high'],  color=color_map[cat], alpha=0.3)

        plt.title(f'{f} Fit with {l} link')
        plt.xlabel("Fire Weather Index -- Including IMERG")
        plt.ylabel("Fire Area Growth km^2")

        #plt.legend(title = pretty_names[c], labels = df.field_response_type.map(pretty_names).unique())
        plt.savefig(f'/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/model_fit_{f}_with_{l}_{y}_func_of_{x}_by_{c}_bic{model.bic}.png', dpi = 900, transparent = False)
        plt.show()
  

## Find the lowest BIC score of the bunch

In [ ]:


stats = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})
stats




In [ ]:
print(stats[stats.Log_Likelyhood.astype("float").min() == stats.Log_Likelyhood.astype("float") ])

print(stats[stats.AIC.astype("float").min() == stats.AIC.astype("float") ])
print(stats[stats.BIC.astype("float").min() == stats.BIC.astype("float") ])



In [ ]:
### Looking at only the GEOS-5

row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)
#row_mask = (~fire3.fireID.str.contains("_")) 

df = fire3[row_mask]

df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling"]].dropna()

y = "farea_diff_rolling"
x = "FWI_rolling"
c = "field_response_type"
#df = df.sort_values(by = x)

# Unique category labels: 'D', 'F', 'G', ...
# color_labels = df[c].unique()

# # List of RGB triplets

# #rgb_values = sns.color_palette("Set1", 3)
# #rgb_values.reverse()

# # Map label to RGB
# color_map = dict(zip(color_labels, rgb_values))

formula = f"{y} ~ {x}:C({c})"
print(formula)

families = ["Gaussian", "Gamma", "Poisson", "NegativeBinomial"]
links = ["Log", "Identity"]
#links = ["Identity"]

ls = []
fs = []
aic = []
ll = []
bic = []
for f in families:
    for l in links: 
        #desc = ModelDesc.from_formula(formula)
        #desc.describe()

        #sm.families.family.Gamma.links
        #link_g = sm.genmod.families.links.Identity()
        #link_g = sm.genmod.families.links.Log()
        #link_g = sm.genmod.families.links.CLogLog()
        #link_g = sm.genmod.families.links.Sqrt()
        #link_g = sm.genmod.families.links.InversePower()
        #link_g = sm.genmod.families.links.NegativeBinomial()
        #link_g = sm.genmod.families.links.Power()
        
        link_g = getattr(sm.genmod.families.links, l)

        method = getattr(sm.families, f)

        model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
        print(model.summary())
        tmp = model.summary2()
        
        fs.append(f)
        ls.append(l)
        aic.append(model.aic)
        ll.append(tmp.tables[0].iloc[2,3])
        bic.append(model.bic)



        df['fitted'] = model.fittedvalues
        df['residuals'] = model.resid_response
        df = df.sort_values(by = x)
        
        # Plot residuals vs fitted values
        res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
        res
        plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
        plt.xlabel('Fitted Values')
        plt.ylabel('Residuals')
        plt.show()
        

        predictions = model.get_prediction(df, transform = True) #df, transform = False
        df['predicted'] = predictions.predicted_mean
        df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T

        actual = sns.scatterplot(x=x, y=y, data=df, hue = c, palette= color_map) # hue = c)
        actual
        handles, labels = actual.get_legend_handles_labels()

        # Customize legend titles and labels
        actual.legend(handles=handles, labels=[ "Full Suppression", "Monitored", "Modified Suppression"], title=pretty_names[c])
        # Plot the fitted values
        pred = sns.lineplot(x=x, y='predicted', data=df, hue = c, palette= color_map, legend = False) #  hue =c,
        pred
        # Plot the confidence intervals
        for cat in df[c].unique():
            #print(cat)
            #print(df.loc[(df[c] == cat), [c]].map(color_map))
            plt.fill_between(df.loc[(df[c] == cat)][x], df.loc[(df[c] == cat)]['conf_int_low'], df.loc[(df[c] == cat)]['conf_int_high'],  color=color_map[cat], alpha=0.3)

        plt.title(f'{f} Fit with {l} link')
        plt.xlabel("Fire Weather Index")
        plt.ylabel("Fire Area Growth km^2")
        #plt.legend()
        plt.savefig(f'/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/model_fit_{f}_with_{l}_{y}_func_of_{x}_by_{c}_bic{model.bic}.png', dpi = 900, transparent = False)
        plt.show()
        
stats_fwi_rolling = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})


print(stats_fwi_rolling[stats_fwi_rolling.Log_Likelyhood.astype("float").min() == stats_fwi_rolling.Log_Likelyhood.astype("float") ])

print(stats_fwi_rolling[stats_fwi_rolling.AIC.astype("float").min() == stats_fwi_rolling.AIC.astype("float") ])
print(stats_fwi_rolling[stats_fwi_rolling.BIC.astype("float").min() == stats_fwi_rolling.BIC.astype("float") ])
stats_fwi_rolling

### Comparing individual models to single-fit model. 

In [ ]:
############ GEO5-5 and IMERGE #################


row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)
#row_mask = (~fire3.fireID.str.contains("_")) 

df = fire3[row_mask]

df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling"]].dropna()

y = "farea_diff_rolling"
x = "GEOS5_IMERGEARLY_rolling"
c = "field_response_type"


formula = f"{y} ~ {x}"
print(formula)

families = ["NegativeBinomial"] ### Chosen based on BIC (and AIC)
links = ["Log"]
#links = ["Identity"]

ls = []
fs = []
aic = []
ll = []
bic = []
for f in families:
    for l in links: 
        #desc = ModelDesc.from_formula(formula)
        #desc.describe()

        #sm.families.family.Gamma.links
        #link_g = sm.genmod.families.links.Identity()
        #link_g = sm.genmod.families.links.Log()
        #link_g = sm.genmod.families.links.CLogLog()
        #link_g = sm.genmod.families.links.Sqrt()
        #link_g = sm.genmod.families.links.InversePower()
        #link_g = sm.genmod.families.links.NegativeBinomial()
        #link_g = sm.genmod.families.links.Power()
        
        link_g = getattr(sm.genmod.families.links, l)

        method = getattr(sm.families, f)

        model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
        print(model.summary())
        tmp = model.summary2()
        
        fs.append(f)
        ls.append(l)
        aic.append(model.aic)
        ll.append(tmp.tables[0].iloc[2,3])
        bic.append(model.bic)

    

        df['fitted'] = model.fittedvalues
        df['residuals'] = model.resid_response
        df = df.sort_values(by = x)
        # Plot residuals vs fitted values
        res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
        res
        plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
        plt.xlabel('Fitted Values')
        plt.ylabel('Residuals')
        plt.show()
        

        predictions = model.get_prediction(df, transform = True) #df, transform = False
        df['predicted'] = predictions.predicted_mean
        df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T

        actual = sns.scatterplot(x=x, y=y, data=df)
        actual
        # Plot the fitted values
        pred = sns.lineplot(x=x, y='predicted', data=df, legend = False)
        pred
        # Plot the confidence intervals
        plt.fill_between(df[x], df['conf_int_low'], df['conf_int_high'], alpha=0.3)

        plt.title(f'{f} Fit with {l} link')
        plt.xlabel(x)
        plt.ylabel(y)
        #plt.legend()
        plt.savefig(f'/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/model_fit_{f}_with_{l}_{y}_func_of_{x}_ONLY_bic{model.bic}.png', dpi = 900, transparent = False)
        plt.show()
        
stats_single_model_imerge = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})

print(f"The model with supression had a BIC of: {stats.BIC.min()}. The model without supression had a BIC of: {stats_single_model_imerge.BIC.min()}")



stats_single_model_imerge

In [ ]:

row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)
#row_mask = (~fire3.fireID.str.contains("_")) 

df = fire3[row_mask]

df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling"]].dropna()

y = "farea_diff_rolling"
x = "FWI_rolling"
c = "field_response_type"


formula = f"{y} ~ {x}"
print(formula)

families = ["NegativeBinomial"] ### Chosen based on BIC (and AIC)
links = ["Identity"]
#links = ["Identity"]

ls = []
fs = []
aic = []
ll = []
bic = []
for f in families:
    for l in links: 
        #desc = ModelDesc.from_formula(formula)
        #desc.describe()

        #sm.families.family.Gamma.links
        #link_g = sm.genmod.families.links.Identity()
        #link_g = sm.genmod.families.links.Log()
        #link_g = sm.genmod.families.links.CLogLog()
        #link_g = sm.genmod.families.links.Sqrt()
        #link_g = sm.genmod.families.links.InversePower()
        #link_g = sm.genmod.families.links.NegativeBinomial()
        #link_g = sm.genmod.families.links.Power()
        
        link_g = getattr(sm.genmod.families.links, l)

        method = getattr(sm.families, f)

        model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
        print(model.summary())
        tmp = model.summary2()
        
        fs.append(f)
        ls.append(l)
        aic.append(model.aic)
        ll.append(tmp.tables[0].iloc[2,3])
        bic.append(model.bic)



        df['fitted'] = model.fittedvalues
        df['residuals'] = model.resid_response

        # Plot residuals vs fitted values
        res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
        res
        plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
        plt.xlabel('Fitted Values')
        plt.ylabel('Residuals')
        plt.show()
        

        predictions = model.get_prediction(df, transform = True) #df, transform = False
        df['predicted'] = predictions.predicted_mean
        df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T
        df = df.sort_values(by = x)

        actual = sns.scatterplot(x=x, y=y, data=df)
        actual
        # Plot the fitted values
        pred = sns.lineplot(x=x, y='predicted', data=df, legend = False)
        pred
        # Plot the confidence intervals
        #plt.fill_between(df['FWI_rolling'], df['conf_int_low'], df['conf_int_high'], c ="field_response_type", alpha=0.3)
        
        plt.fill_between(df['FWI_rolling'], df['conf_int_low'], df['conf_int_high'], alpha=0.3)

        plt.title(f'{f} Fit with {l} link')
        plt.xlabel(x)
        plt.ylabel(y)
        #plt.legend()
        plt.savefig(f'/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/model_fit_{f}_with_{l}_{y}_func_of_{x}_ONLY_bic{model.bic}.png', dpi = 900, transparent = False)
        plt.show()
        
stats_single_model_fwi_rolling = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})

print(f"The model with supression had a BIC of: {stats_fwi_rolling.BIC.min()}. The model without supression had a BIC of: {stats_single_model_fwi_rolling.BIC.min()}")



stats_single_model_fwi_rolling

In [ ]:
df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
       'field_response_type', 
       'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling",
       'fitted', 'residuals']].dropna()
predictions = model.get_prediction(df)
df['predicted'] = predictions.predicted_mean
df['conf_int_low'], df['conf_int_high'] = predictions.conf_int(method = "delta").T

df = df.sort_values(by = 'FWI_rolling')

actual = sns.scatterplot(x='FWI_rolling', y='farea_diff_rolling', data=df)
actual
# Plot the fitted values
pred = sns.lineplot(x='FWI_rolling', y='predicted', data=df, legend = False)
pred
# Plot the confidence intervals
plt.fill_between(df['FWI_rolling'], df['conf_int_low'], df['conf_int_high'], alpha=0.3)

plt.title('GLM Predictions with Confidence Intervals')
plt.xlabel('FWI_rolling')
plt.ylabel('farea_diff_rolling')
#plt.legend()
plt.show()

In [ ]:
#plt.scatter(df[x], df[y], hue = df[c])

#fire3.columns

## Plots of the best-fit models

In [ ]:
#### GEOS-5 ONLY

row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)
#row_mask = (~fire3.fireID.str.contains("_")) 

df = fire3[row_mask]

df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling"]].dropna()

y = "farea_diff_rolling"
x = "FWI_rolling"
c = "field_response_type"
#df = df.sort_values(by = x)

# Unique category labels: 'D', 'F', 'G', ...
# color_labels = df[c].unique()

# # List of RGB triplets

# #rgb_values = sns.color_palette("Set1", 3)
# #rgb_values.reverse()

# # Map label to RGB
# color_map = dict(zip(color_labels, rgb_values))

formula = f"{y} ~ {x}:C({c})"


families = ["NegativeBinomial"] ### Chosen based on BIC (and AIC)
links = ["Identity"]
#links = ["Identity"]

ls = []
fs = []
aic = []
ll = []
bic = []
for f in families:
    for l in links: 
        #desc = ModelDesc.from_formula(formula)
        #desc.describe()

        #sm.families.family.Gamma.links
        #link_g = sm.genmod.families.links.Identity()
        #link_g = sm.genmod.families.links.Log()
        #link_g = sm.genmod.families.links.CLogLog()
        #link_g = sm.genmod.families.links.Sqrt()
        #link_g = sm.genmod.families.links.InversePower()
        #link_g = sm.genmod.families.links.NegativeBinomial()
        #link_g = sm.genmod.families.links.Power()
        
        link_g = getattr(sm.genmod.families.links, l)

        method = getattr(sm.families, f)

        model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
        print(model.summary())
        tmp = model.summary2()
        
        fs.append(f)
        ls.append(l)
        aic.append(model.aic)
        ll.append(tmp.tables[0].iloc[2,3])
        bic.append(model.bic)



        df['fitted'] = model.fittedvalues
        df['residuals'] = model.resid_response
        df = df.sort_values(by = x)
        
        # Plot residuals vs fitted values
        res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
        res
        plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
        plt.xlabel('Fitted Values')
        plt.ylabel('Residuals')
        plt.show()
        

        predictions = model.get_prediction(df, transform = True) #df, transform = False
        df['predicted'] = predictions.predicted_mean
        df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T

        actual = sns.scatterplot(x=x, y=y, data=df, hue = c, palette= color_map) # hue = c)
        actual
        handles, labels = actual.get_legend_handles_labels()

        # Customize legend titles and labels
        actual.legend(handles=handles, labels=[ "Full Suppression", "Monitored", "Modified Suppression"], title=pretty_names[c])
        # Plot the fitted values
        pred = sns.lineplot(x=x, y='predicted', data=df, hue = c, palette= color_map, legend = False) #  hue =c,
        pred
        # Plot the confidence intervals
        for cat in df[c].unique():
            #print(cat)
            #print(df.loc[(df[c] == cat), [c]].map(color_map))
            plt.fill_between(df.loc[(df[c] == cat)][x], df.loc[(df[c] == cat)]['conf_int_low'], df.loc[(df[c] == cat)]['conf_int_high'],  color=color_map[cat], alpha=0.3)

        #plt.title(f'{f} Fit with {l} link')
        plt.title('')
        plt.xlabel("Fire Weather Index -- GEOS5 Only")
        plt.ylabel("Fire Area Growth km^2")
        #plt.legend()
        plt.savefig(f'/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/Pretty_model_fit_{f}_with_{l}_{y}_func_of_{x}_via_{c}_bic{model.bic}.png', dpi = 900, transparent = False)
        plt.show()
        
stats_single_model_fwi_rolling = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})

print(f"The model with supression had a BIC of: {stats_fwi_rolling.BIC.min()}. The model without supression had a BIC of: {stats_single_model_fwi_rolling.BIC.min()}")

In [ ]:
#### IMERG

row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)
#row_mask = (~fire3.fireID.str.contains("_")) 

df = fire3[row_mask]

df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling"]].dropna()

y = "farea_diff_rolling"
x = "GEOS5_IMERGEARLY_rolling"
c = "field_response_type"
#df = df.sort_values(by = x)

# Unique category labels: 'D', 'F', 'G', ...
# color_labels = df[c].unique()

# # List of RGB triplets

# #rgb_values = sns.color_palette("Set1", 3)
# #rgb_values.reverse()

# # Map label to RGB
# color_map = dict(zip(color_labels, rgb_values))

formula = f"{y} ~ {x}:C({c})"


families = ["NegativeBinomial"] ### Chosen based on BIC (and AIC)
links = ["Log"]
#links = ["Identity"]

ls = []
fs = []
aic = []
ll = []
bic = []
for f in families:
    for l in links: 
        #desc = ModelDesc.from_formula(formula)
        #desc.describe()

        #sm.families.family.Gamma.links
        #link_g = sm.genmod.families.links.Identity()
        #link_g = sm.genmod.families.links.Log()
        #link_g = sm.genmod.families.links.CLogLog()
        #link_g = sm.genmod.families.links.Sqrt()
        #link_g = sm.genmod.families.links.InversePower()
        #link_g = sm.genmod.families.links.NegativeBinomial()
        #link_g = sm.genmod.families.links.Power()
        
        link_g = getattr(sm.genmod.families.links, l)

        method = getattr(sm.families, f)

        model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
        print(model.summary())
        tmp = model.summary2()
        
        fs.append(f)
        ls.append(l)
        aic.append(model.aic)
        ll.append(tmp.tables[0].iloc[2,3])
        bic.append(model.bic)



        df['fitted'] = model.fittedvalues
        df['residuals'] = model.resid_response
        df = df.sort_values(by = x)
        
        # Plot residuals vs fitted values
        res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
        res
        plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
        plt.xlabel('Fitted Values')
        plt.ylabel('Residuals')
        plt.show()
        

        predictions = model.get_prediction(df, transform = True) #df, transform = False
        df['predicted'] = predictions.predicted_mean
        df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T

        actual = sns.scatterplot(x=x, y=y, data=df, hue = c, palette= color_map) # hue = c)
        actual
        handles, labels = actual.get_legend_handles_labels()

        # Customize legend titles and labels
        actual.legend(handles=handles, labels=[ "Full Suppression", "Monitored", "Modified Suppression"], title=pretty_names[c])
        # Plot the fitted values
        pred = sns.lineplot(x=x, y='predicted', data=df, hue = c, palette= color_map, legend = False) #  hue =c,
        pred
        # Plot the confidence intervals
        for cat in df[c].unique():
            #print(cat)
            #print(df.loc[(df[c] == cat), [c]].map(color_map))
            plt.fill_between(df.loc[(df[c] == cat)][x], df.loc[(df[c] == cat)]['conf_int_low'], df.loc[(df[c] == cat)]['conf_int_high'],  color=color_map[cat], alpha=0.3)

        #plt.title(f'{f} Fit with {l} link')
        plt.title('')
        plt.xlabel("Fire Weather Index -- including IMERG")
        plt.ylabel("Fire Area Growth km^2")
        #plt.legend()
        #plt.savefig(f'/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/Pretty_model_fit_{f}_with_{l}_{y}_func_of_{x}_via_{c}_bic{model.bic}.png', dpi = 900, transparent = False)
        plt.show()
        
stats_single_model_fwi_rolling = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})

print(f"The model with supression had a BIC of: {stats_fwi_rolling.BIC.min()}. The model without supression had a BIC of: {stats_single_model_fwi_rolling.BIC.min()}")

# Supplemental Exploratory plots

- Supression by day-of-fire

- Supression vs fuels for explaining stuff

In [ ]:
import matplotlib.colors as mcolors

row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires))  & (fire3.field_response_type == "FUL")



scatter = plt.scatter(fire3[row_mask]["GEOS-5.IMERGEARLY_rolling"], fire3[row_mask].farea_diff_rolling, c = fire3[row_mask].day_of_fire,  norm=mcolors.LogNorm())
colorbar = plt.colorbar(scatter)
colorbar.set_label('Day of Fire')
#plt.legend()
plt.title("Fully Supressed Fires by Day since Fire Ignition")
plt.ylabel("Fire Area Growth km^2")
plt.xlabel("Fire Weather Index -- Including IMERG")
plt.savefig('/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/Supressed_growth_vs_FWI_IMERGE.png', dpi = 900, transparent = False)
plt.show()

In [ ]:
row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires))  & (fire3.field_response_type == "FUL")



scatter = plt.scatter(fire3[row_mask]["FWI_rolling"], fire3[row_mask].farea_diff_rolling, c = fire3[row_mask].day_of_fire,  norm=mcolors.LogNorm() ) # 
colorbar = plt.colorbar(scatter)
colorbar.set_label('Day of Fire')
#plt.legend()
plt.title("Fully Supressed Fires by Day since Fire Ignition")
plt.ylabel("Fire Area Growth km^2")
plt.xlabel("Fire Weather Index -- GEOS5 Only")
plt.savefig('/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/Supressed_growth_vs_FWI_GEOS5.png', dpi = 900, transparent = False)
plt.show()

# Getting Fuels vs supression status

In [ ]:
#gf = rio.open_rasterio("/projects/old_shared/fire_weather_vis/Lightning_analysis/Plant_functional_types/NFI_MODIS250m_2011_kNN_LandCover_VegNonTreed_v1.tif")

#fuels = xr.open_dataset("/projects/old_shared/fire_weather_vis/Lightning_analysis/Plant_functional_types/NFI_MODIS250m_2011_kNN_LandCover_VegNonTreed_v1.tif")
#fuels.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

gf = xr.open_dataset("/projects/old_shared/fire_weather_vis/Lightning_analysis/Plant_functional_types/ESACCI-LC-L4-PFT-Map-300m-P1Y-2020-v2.0.8.nc", engine='netcdf4') #### NOT LOADING FOR SOME REASON

In [ ]:
import pyproj

#fuels = gf.to_dataset('band')

#fuels = fuels.rename({1: 'pft'})
#lat = gf['lat']
#lon = gf['lon']

# If the dataset contains the necessary attributes to define a CRS, use them to create a CRS object
# Example assumes WGS84 latitude and longitude
#crs = pyproj.CRS("EPSG:4326")


#gf = gf.assign_coords(lat=gf["lat"] , lon=gf['lon'])
#gf.rio.write_crs(crs, inplace=True)


fire = fire3.to_crs("EPSG:4326")




In [ ]:
str(round(fire.bounds.minx.min(),1))

min_y = round(fire.bounds.miny.min(),1)
max_y = round(fire.bounds.maxy.max(), 1)

print(min_y)
print(max_y)

In [ ]:
fuels = gf.sel(lon=slice(float(round(fire.bounds.minx.min(),1)), float(round(fire.bounds.maxx.max(),1))), 
               lat=slice(max_y, min_y))

In [ ]:
fuels

In [ ]:
#gf

In [ ]:
fuels["GRASS-NAT"].plot()

In [ ]:
#fuels = fuels.rio.to_crs(fire3.crs)

# fire = fire3.to_crs(fuels.rio.crs)

# fuels = fuels.sel(x=slice(fire3.bounds.minx.min(), fire3.bounds.maxx.max()), y=slice(fire3.bounds.miny.min(), fire3.bounds.maxy.max()))

#fuels = fuels.rename({"x" : "lat", "y": "lon"})

In [ ]:
print(gf.rio.bounds())

print(fire.bounds.minx.min(), fire.bounds.maxx.max(), fire.bounds.miny.min(), fire.bounds.maxy.max())

In [ ]:
gf

In [ ]:
#fire.geometry[0]

In [ ]:
fuels.data_vars

In [ ]:
gf

In [ ]:
#fuels = fuels["WATER"].rio.write_crs(fire.crs)


#fuels["WATER"] = fuels["WATER"].rio.write_crs("EPSG:4326")

# fuels.rio.clip([fire.geometry[0]])


#clipped = fuels.rio.clip(fire.geometry)

#clipped = fuels[["WATER"]].rio.clip(fire.geometry)

In [ ]:
#fuels["WATER"].rio.write_crs(fire.crs)

In [ ]:
# print(type(fuels))

# print(type(fuels["WATER"]))

In [ ]:
from rasterio import features
from shapely.geometry import mapping

landcover_vars = ['BARE', 'BUILT', 'GRASS-MAN', 'GRASS-NAT', 'SHRUBS-BD', 'SHRUBS-BE', 'SHRUBS-ND', 'SHRUBS-NE', 'WATER_INLAND', 'SNOWICE', 'TREES-BD', 'TREES-BE', 'TREES-ND', 'TREES-NE']
#print(landcover_vars)

In [ ]:
fuels

In [ ]:
fuels = fuels.chunk({'lat': 512, 'lon': 512})

In [ ]:
### Subsetting fires to just final perimeter to cut down on compute. Something is wrong though

# fire["is_final"] = np.nan

def get_final(df):
    df["is_final"] = (df.t == df.t.max())
    return(df)

# fire = fire.groupby("fireID").apply(get_final).reset_index(drop=True )

# fire = fire[fire.is_final]

In [ ]:
row_mask = (~fire.fireID.str.contains("_")) & (fire.farea_diff > 0.1) & (fire.fireID.isin(long_fires))

fire = fire[row_mask]

In [ ]:
len(fire.fireID.unique())

In [ ]:
len(fire)

In [ ]:
#fire[fire.fireID == '10056'].geometry

In [ ]:
# Decided this was complicated and unlikely to be helpful. 
# def assign_differences():
#     df['diff_geometry']
#     ### Find the timesteps that need to be differenced. 
    
    
#     ### Take the difference of each
    
    
#     ### assign to column


# fire[fire.fireID == '10056'].geometry[23].difference(fire[fire.fireID == '10056'].geometry[36])

In [ ]:
#### Extraction ---- do not rerun if you can help it. 

# def rasterize_geometry(geometry, xarray_data):
#     transform = xarray_data.rio.transform()
#     out_shape = xarray_data.rio.shape
#     return features.geometry_mask([mapping(geometry)], transform=transform, invert=True, out_shape=out_shape)

# # Initialize a list to store the dominant landcover types
# dominant_landcovers = []

# # Loop through each geometry in the GeoDataFrame
# itr = 0
# for geometry in fire.geometry:
#     itr = itr + 1
#     print(itr/len(fire))
#     # Rasterize the geometry to create a mask
#     mask = rasterize_geometry(geometry, fuels[landcover_vars[0]])
    
#     # Initialize a dictionary to store the sum of each landcover type within the geometry
#     landcover_sums = {var: 0 for var in landcover_vars}
    
#     # Loop through each landcover variable
#     for var in landcover_vars:
#         # Access the DataArray corresponding to the current landcover variable
#         data_array = fuels[var]
        
#         # Apply the mask to the current landcover data variable using xr.where
#         masked_landcover = data_array.where(mask)
        
#         # Sum the values within the masked area and store in the dictionary
#         #landcover_sums[var] = masked_landcover.sum().item()
#         landcover_sums[var] = masked_landcover.sum().compute().item()
    
#     # Determine the dominant landcover type (the one with the highest sum)
#     dominant_landcover = max(landcover_sums, key=landcover_sums.get)
#     dominant_landcovers.append(dominant_landcover)

# # Add the dominant landcover types to the GeoDataFrame
# fire['dominant_landcover'] = dominant_landcovers

# # Save or further process the GeoDataFrame
# #gdf.to_file('path_to_save_geodataframe.shp')

# print(fire[['geometry', 'dominant_landcover']])

# # def get_dominant_value(geometry, raster):
# #     # Clip the raster with the geometry
# #     clipped = raster.rio.clip([geometry])
    
# #     # Get the raster values within the geometry
# #     values = clipped.values.flatten()
    
# #     # Remove NaN values
# #     values = values[~np.isnan(values)]
    
# #     # Find the most common value
# #     if len(values) > 0:
# #         most_common_value = Counter(values).most_common(1)[0][0]
# #     else:
# #         most_common_value = np.nan
    
# #     return most_common_value

# # # # Apply the function to each geometry in the vector dataset
# # fire['dominant_raster_value'] = fire.geometry.apply(lambda geom: get_dominant_value(geom, fuels))

# # #gf.sel(lon = slice(round(fire.bounds.minx.min(),1), round(fire.bounds.maxx.max(),1)), lat = slice(46, 58))#.mean(dim='time').plot()

In [ ]:
#len(fire.dominant_landcover)

In [ ]:
#len(fire.field_response_type)

In [ ]:
#fire.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/landcover_merged_datasets/fires_merged_ESACCI-LC-L4-PFT-Map-300m-P1Y-2020-v2.0.8.csv")

In [ ]:
#[*color_labels]

In [ ]:
fire = pd.read_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/landcover_merged_datasets/fires_merged_ESACCI-LC-L4-PFT-Map-300m-P1Y-2020-v2.0.8.csv")

### With landcover extracted, checking statistical models 

In [ ]:
### Looking at only the GEOS-5

#row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)
#row_mask = (~fire3.fireID.str.contains("_")) 

df = fire

df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling", "dominant_landcover"]].dropna()


df = df[df.dominant_landcover != "SHRUBS-NE"] ## Excluding because only one fire was majority shubs

print(f"Excluding shrubs lost {len(df[df.dominant_landcover == 'SHRUBS-NE'])}")

y = "farea_diff_rolling"
x = "FWI_rolling"
c = "dominant_landcover"
df = df.sort_values(by = x)

#Unique category labels: 'D', 'F', 'G', ...
color_labels = df[c].unique()

# List of RGB triplets

rgb_values = sns.color_palette("colorblind", 2)
#rgb_values.reverse()

# Map label to RGB
color_map = dict(zip(color_labels, rgb_values))

formula = f"{y} ~ {x}:C({c})"
print(formula)

families = ["Gaussian", "Gamma", "Poisson", "NegativeBinomial"]
links = ["Log", "Identity"]
#links = ["Identity"]

ls = []
fs = []
aic = []
ll = []
bic = []
for f in families:
    for l in links: 
        #desc = ModelDesc.from_formula(formula)
        #desc.describe()

        #sm.families.family.Gamma.links
        #link_g = sm.genmod.families.links.Identity()
        #link_g = sm.genmod.families.links.Log()
        #link_g = sm.genmod.families.links.CLogLog()
        #link_g = sm.genmod.families.links.Sqrt()
        #link_g = sm.genmod.families.links.InversePower()
        #link_g = sm.genmod.families.links.NegativeBinomial()
        #link_g = sm.genmod.families.links.Power()
        
        link_g = getattr(sm.genmod.families.links, l)

        method = getattr(sm.families, f)

        model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
        print(model.summary())
        tmp = model.summary2()
        
        fs.append(f)
        ls.append(l)
        aic.append(model.aic)
        ll.append(tmp.tables[0].iloc[2,3])
        bic.append(model.bic)



        df['fitted'] = model.fittedvalues
        df['residuals'] = model.resid_response
        df = df.sort_values(by = x)
        
        # Plot residuals vs fitted values
        res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
        res
        plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
        plt.xlabel('Fitted Values')
        plt.ylabel('Residuals')
        plt.show()
        

        predictions = model.get_prediction(df, transform = True) #df, transform = False
        df['predicted'] = predictions.predicted_mean
        df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T

        actual = sns.scatterplot(x=x, y=y, data=df, hue = c, palette= color_map) # hue = c)
        actual
        #handles, labels = actual.get_legend_handles_labels()

        # Customize legend titles and labels
        #actual.legend(handles=handles, labels= [*color_labels], title= c)
        # Plot the fitted values
        pred = sns.lineplot(x=x, y='predicted', data=df, hue = c, palette= color_map, legend = False) #  hue =c,
        pred
        # Plot the confidence intervals
        for cat in df[c].unique():
            #print(cat)
            #print(df.loc[(df[c] == cat), [c]].map(color_map))
            plt.fill_between(df.loc[(df[c] == cat)][x], df.loc[(df[c] == cat)]['conf_int_low'], df.loc[(df[c] == cat)]['conf_int_high'],  color=color_map[cat], alpha=0.3)

        plt.title(f'{f} Fit with {l} link')
        plt.xlabel("Fire Weather Index")
        plt.ylabel("Fire Area Growth km^2")
        #plt.legend()
        plt.savefig(f'/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/model_fit_{f}_with_{l}_{y}_func_of_{x}_by_{c}_bic{model.bic}.png', dpi = 900, transparent = False)
        plt.show()
        
stats_fwi_rolling = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})


print(stats_fwi_rolling[stats_fwi_rolling.Log_Likelyhood.astype("float").min() == stats_fwi_rolling.Log_Likelyhood.astype("float") ])

print(stats_fwi_rolling[stats_fwi_rolling.AIC.astype("float").min() == stats_fwi_rolling.AIC.astype("float") ])
print(stats_fwi_rolling[stats_fwi_rolling.BIC.astype("float").min() == stats_fwi_rolling.BIC.astype("float") ])
stats_fwi_rolling

In [ ]:
### Now check IMERG data


df = fire

df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling", "dominant_landcover"]].dropna()


df = df[df.dominant_landcover != "SHRUBS-NE"] ## Excluding because only one fire was majority shubs

print(f"Excluding shrubs lost {len(df[df.dominant_landcover == 'SHRUBS-NE'])}")

y = "farea_diff_rolling"
x = "GEOS5_IMERGEARLY_rolling"
c = "dominant_landcover"
# df = df.sort_values(by = x)

# #Unique category labels: 'D', 'F', 'G', ...
# color_labels = df[c].unique()

# # List of RGB triplets

# rgb_values = sns.color_palette("colorblind", 2)
# #rgb_values.reverse()

# # Map label to RGB
# color_map = dict(zip(color_labels, rgb_values))

formula = f"{y} ~ {x}:C({c})"
print(formula)

families = ["Gaussian", "Gamma", "Poisson", "NegativeBinomial"]
links = ["Log", "Identity"]
#links = ["Identity"]

ls = []
fs = []
aic = []
ll = []
bic = []
for f in families:
    for l in links: 
        #desc = ModelDesc.from_formula(formula)
        #desc.describe()

        #sm.families.family.Gamma.links
        #link_g = sm.genmod.families.links.Identity()
        #link_g = sm.genmod.families.links.Log()
        #link_g = sm.genmod.families.links.CLogLog()
        #link_g = sm.genmod.families.links.Sqrt()
        #link_g = sm.genmod.families.links.InversePower()
        #link_g = sm.genmod.families.links.NegativeBinomial()
        #link_g = sm.genmod.families.links.Power()
        
        link_g = getattr(sm.genmod.families.links, l)

        method = getattr(sm.families, f)

        model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
        print(model.summary())
        tmp = model.summary2()
        
        fs.append(f)
        ls.append(l)
        aic.append(model.aic)
        ll.append(tmp.tables[0].iloc[2,3])
        bic.append(model.bic)



        df['fitted'] = model.fittedvalues
        df['residuals'] = model.resid_response
        df = df.sort_values(by = x)
        
        # Plot residuals vs fitted values
        res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
        res
        plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
        plt.xlabel('Fitted Values')
        plt.ylabel('Residuals')
        plt.show()
        

        predictions = model.get_prediction(df, transform = True) #df, transform = False
        df['predicted'] = predictions.predicted_mean
        df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T

        actual = sns.scatterplot(x=x, y=y, data=df, hue = c, palette= color_map) # hue = c)
        actual
        #handles, labels = actual.get_legend_handles_labels()

        # Customize legend titles and labels
        #actual.legend(handles=handles, labels= [*color_labels], title= c)
        # Plot the fitted values
        pred = sns.lineplot(x=x, y='predicted', data=df, hue = c, palette= color_map, legend = False) #  hue =c,
        pred
        # Plot the confidence intervals
        for cat in df[c].unique():
            #print(cat)
            #print(df.loc[(df[c] == cat), [c]].map(color_map))
            plt.fill_between(df.loc[(df[c] == cat)][x], df.loc[(df[c] == cat)]['conf_int_low'], df.loc[(df[c] == cat)]['conf_int_high'],  color=color_map[cat], alpha=0.3)

        plt.title(f'{f} Fit with {l} link')
        plt.xlabel("Fire Weather Index")
        plt.ylabel("Fire Area Growth km^2")
        #plt.legend()
        plt.savefig(f'/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/model_fit_{f}_with_{l}_{y}_func_of_{x}_by_{c}_bic{model.bic}.png', dpi = 900, transparent = False)
        plt.show()
        
stats_fwi_rolling = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})


print(stats_fwi_rolling[stats_fwi_rolling.Log_Likelyhood.astype("float").min() == stats_fwi_rolling.Log_Likelyhood.astype("float") ])

print(stats_fwi_rolling[stats_fwi_rolling.AIC.astype("float").min() == stats_fwi_rolling.AIC.astype("float") ])
print(stats_fwi_rolling[stats_fwi_rolling.BIC.astype("float").min() == stats_fwi_rolling.BIC.astype("float") ])
stats_fwi_rolling

# Becuase it seems like the fuel catagorization *just barely* looses to the supression chariterization, making a matrix to look at if supression and fuels are litterally the same catagory

It could be that the models of supression *only* beat out the the fuel model because I had to exclude 1 point, which would be wild. 

In [ ]:
supression = [*fire.field_response_type.unique()]
land_cover = [*fire.dominant_landcover.unique()]

#common_ids = ['10972', '10896', '12656'] ## derived below



cat = []
ln = []
size = []
ids = []
ids_ln = []

cm_10972 = []
cm_10896 = []
cm_12656 = []



for s in supression:
    for l in land_cover:
        
        length = len(df[(df.dominant_landcover == l) & (df.field_response_type == s)])
        sub = df[(df.dominant_landcover == l) & (df.field_response_type == s)]
        ids.append(sub.fireID.unique())
        cat.append(f"{s}_{l}")
        ln.append(length)
        size.append(sub.farea_diff_rolling.mean())
        ids_ln.append(len(sub.fireID.unique()))
        cm_10972.append(len(sub[sub.fireID == '10972']))
        cm_10896.append(len(sub[sub.fireID == '10896']))
        cm_12656.append(len(sub[sub.fireID == '12656']))
        
        if length > 0:
            print(f" Fire-days with {s} response and {l} cover type: {length}")
            
for s in supression:
    for l in land_cover:
        length = len(df[(df.dominant_landcover == l) & (df.field_response_type == s)])
        sub = df[(df.dominant_landcover == l) & (df.field_response_type == s)]
        if length > 0:
            print(f" {s} with {l} {sub.fireID.unique()}")

for s in supression:
    for l in land_cover:
        length = len(df[(df.dominant_landcover == l) & (df.field_response_type == s)])
        sub = df[(df.dominant_landcover == l) & (df.field_response_type == s)]
        if length > 0:
            print(f" {s} with {l} {sub.farea_diff_rolling.mean()}")

In [ ]:
quick_stats = pd.DataFrame({"catagory" : cat, "n": ln, "size": size, "fireID": ids, "num_fires": ids_ln, "from_10972": cm_10972,"from_10896": cm_10896, "from_12656": cm_12656 })

quick_stats

In [ ]:
fire.columns

In [ ]:
fire[['fireID', 't', 
       'n_pixels', 'n_newpixels', 'farea', 'fperim', 'flinelen', 'duration',
       'pixden', 'meanFRP', 'field_system_fire_cause',
       'field_response_type','farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"FWI_rolling",
       'dominant_landcover']][fire.fireID == '10972']  ### First two days started with trees, spread to grass. Monitored. There was a big transition in the FWI(s) at the fuel transition period, as well as a jump in area. 

In [ ]:
fire[['fireID', 't', 
       'n_pixels', 'n_newpixels', 'farea', 'fperim', 'flinelen', 'duration',
       'pixden', 'meanFRP', 'field_system_fire_cause',
       'field_response_type','farea_diff_rolling', "FWI_rolling", 'GEOS-5.IMERGEARLY_rolling',
       'dominant_landcover']][fire.fireID == '10896'] ### first 4 days stared with grass, spread to trees. Was a jump in area but no jump in either FWI(s). 

In [ ]:
fire[['fireID', 't', 
       'n_pixels', 'n_newpixels', 'farea', 'fperim', 'flinelen', 'duration',
       'pixden', 'meanFRP', 'field_system_fire_cause',
       'field_response_type','farea_diff_rolling', "FWI_rolling", 'GEOS-5.IMERGEARLY_rolling',
       'dominant_landcover']][fire.fireID == '12656'] ### First 5 days in trees, then spread to grass. Also no jump in FWI but a jump in area. 

In [ ]:
# def common_member(a, b):
#     #print(a)
#     a_set = set(a)
#     b_set = set(b)
     
#     # check length 
#     if len(a_set.intersection(b_set)) > 0:
#         return(a_set.intersection(b_set))  
#     else:
#         return(np.nan)

In [ ]:
# ## Check if there are any repeat fires

# for c in quick_stats.catagory.unique():
#     for c2 in quick_stats.catagory.unique():
#         if(c != c2):
#             print(f"{c} and {c2} both have : {common_member(*quick_stats[quick_stats.catagory == c].fireID, *quick_stats[quick_stats.catagory == c2].fireID)}")

# Check if the supression is still the best explanatory variable even when only looking at the same fuel type

 The "FUL" catagory has the least differentiation between the two. There is only one fire that has the grasses as the dominant landcover type. 
 
 


In [ ]:
fuel_cat = "TREES-NE"

df = fire[fire.dominant_landcover == fuel_cat]

df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling", "dominant_landcover"]].dropna()




y = "farea_diff_rolling"
x = "GEOS5_IMERGEARLY_rolling"
c = "field_response_type"

df = df.sort_values(by = x)

pretty_names = {c: "Response Type", "MON": "Monitored", "FUL": "Full Supression", "MOD": "Modified Supression"}

# Unique category labels: 'D', 'F', 'G', ...
color_labels = df[c].unique()

# List of RGB triplets
rgb_values = sns.color_palette("Set1", 3)
rgb_values.reverse()

# Map label to RGB
color_map = dict(zip(color_labels, rgb_values))

# # Finally use the mapped values
# plt.scatter(df['carat'], df['price'], c=df[c].map(color_map))


formula = f"{y} ~ {x}:C({c})"
print(formula)

families = ["Gaussian", "Gamma", "Poisson", "NegativeBinomial"]
links = ["Log", "Identity"]
#links = ["Identity"]

ls = []
fs = []
aic = []
ll = []
bic = []
for f in families:
    for l in links: 
        #desc = ModelDesc.from_formula(formula)
        #desc.describe()

        #sm.families.family.Gamma.links
        #link_g = sm.genmod.families.links.Identity()
        #link_g = sm.genmod.families.links.Log()
        #link_g = sm.genmod.families.links.CLogLog()
        #link_g = sm.genmod.families.links.Sqrt()
        #link_g = sm.genmod.families.links.InversePower()
        #link_g = sm.genmod.families.links.NegativeBinomial()
        #link_g = sm.genmod.families.links.Power()
        
        link_g = getattr(sm.genmod.families.links, l)

        method = getattr(sm.families, f)

        model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
        print(model.summary())
        tmp = model.summary2()
        
        fs.append(f)
        ls.append(l)
        aic.append(model.aic)
        ll.append(tmp.tables[0].iloc[2,3])
        bic.append(model.bic)



        df['fitted'] = model.fittedvalues
        df['residuals'] = model.resid_response
        df = df.sort_values(by = x)
        # Plot residuals vs fitted values
        res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
        res
        plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
        plt.xlabel('Fitted Values')
        plt.ylabel('Residuals')
        plt.show()
        
    
        predictions = model.get_prediction(df, transform = True) #df, transform = False
        df['predicted'] = predictions.predicted_mean
        df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T

        actual = sns.scatterplot(x=x, y=y, data=df, hue = c, palette = color_map) #facet_kws={'legend_out': True}
        
        handles, labels = actual.get_legend_handles_labels()

        # Customize legend titles and labels
        actual.legend(handles=handles, labels=["Full Suppression", "Monitored",  "Modified Suppression"], title=pretty_names[c])
        # actual._legend.set_title(pretty_names[c])
        #  # replace labels
        # for t, l in zip(actual._legend.texts, df.field_response_type.map(pretty_names).unique()):
        #     t.set_text(l)
        #Plot the fitted values
        pred = sns.lineplot(x=x, y='predicted', data=df, hue =c, legend = False, palette = color_map)
        pred
        # Plot the confidence intervals
        for cat in df[c].unique():
            #print(cat)
            #print(df.loc[(df[c] == cat), [c]].map(color_map))
            plt.fill_between(df.loc[(df[c] == cat)][x], df.loc[(df[c] == cat)]['conf_int_low'], df.loc[(df[c] == cat)]['conf_int_high'],  color=color_map[cat], alpha=0.3)

        plt.title(f'{f} Fit with {l} link')
        plt.xlabel("Fire Weather Index -- Including IMERG")
        plt.ylabel("Fire Area Growth km^2")

        #plt.legend(title = pretty_names[c], labels = df.field_response_type.map(pretty_names).unique())
        plt.savefig(f'/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/subsetted_to_{fuel_cat}_model_fit_{f}_with_{l}_{y}_func_of_{x}_by_{c}_bic{model.bic}.png', dpi = 900, transparent = False)
        plt.show()
        
stats_fwi_rolling = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})


print(stats_fwi_rolling[stats_fwi_rolling.Log_Likelyhood.astype("float").min() == stats_fwi_rolling.Log_Likelyhood.astype("float") ])

print(stats_fwi_rolling[stats_fwi_rolling.AIC.astype("float").min() == stats_fwi_rolling.AIC.astype("float") ])
print(stats_fwi_rolling[stats_fwi_rolling.BIC.astype("float").min() == stats_fwi_rolling.BIC.astype("float") ])


stats_fwi_rolling

In [ ]:
fuel_cat = "TREES-NE"

df = fire[fire.dominant_landcover == fuel_cat]

df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling", "dominant_landcover"]].dropna()


df = df.sort_values(by = x)

y = "farea_diff_rolling"
x = "FWI_rolling"
c = "field_response_type"

pretty_names = {c: "Response Type", "MON": "Monitored", "FUL": "Full Supression", "MOD": "Modified Supression"}

# Unique category labels: 'D', 'F', 'G', ...
color_labels = df[c].unique()

# List of RGB triplets
rgb_values = sns.color_palette("Set1", 3)
rgb_values.reverse()

# Map label to RGB
color_map = dict(zip(color_labels, rgb_values))

# # Finally use the mapped values
# plt.scatter(df['carat'], df['price'], c=df[c].map(color_map))


formula = f"{y} ~ {x}:C({c})"
print(formula)

families = ["Gaussian", "Gamma", "Poisson", "NegativeBinomial"]
links = ["Log", "Identity"]
#links = ["Identity"]

ls = []
fs = []
aic = []
ll = []
bic = []
for f in families:
    for l in links: 
        #desc = ModelDesc.from_formula(formula)
        #desc.describe()

        #sm.families.family.Gamma.links
        #link_g = sm.genmod.families.links.Identity()
        #link_g = sm.genmod.families.links.Log()
        #link_g = sm.genmod.families.links.CLogLog()
        #link_g = sm.genmod.families.links.Sqrt()
        #link_g = sm.genmod.families.links.InversePower()
        #link_g = sm.genmod.families.links.NegativeBinomial()
        #link_g = sm.genmod.families.links.Power()
        
        link_g = getattr(sm.genmod.families.links, l)

        method = getattr(sm.families, f)

        model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
        print(model.summary())
        tmp = model.summary2()
        
        fs.append(f)
        ls.append(l)
        aic.append(model.aic)
        ll.append(tmp.tables[0].iloc[2,3])
        bic.append(model.bic)



        # df['fitted'] = model.fittedvalues
        # df['residuals'] = model.resid_response
        # df = df.sort_values(by = x)
        # # Plot residuals vs fitted values
        # res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
        # res
        # plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
        # plt.xlabel('Fitted Values')
        # plt.ylabel('Residuals')
        # plt.show()
        
    
        predictions = model.get_prediction(df, transform = True) #df, transform = False
        df['predicted'] = predictions.predicted_mean
        df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T

        actual = sns.scatterplot(x=x, y=y, data=df, hue = c, palette = color_map) #facet_kws={'legend_out': True}
        
        handles, labels = actual.get_legend_handles_labels()

        # Customize legend titles and labels
        actual.legend(handles=handles, labels=["Full Suppression", "Monitored",  "Modified Suppression"], title=pretty_names[c])
        # actual._legend.set_title(pretty_names[c])
        #  # replace labels
        # for t, l in zip(actual._legend.texts, df.field_response_type.map(pretty_names).unique()):
        #     t.set_text(l)
        #Plot the fitted values
        pred = sns.lineplot(x=x, y='predicted', data=df, hue =c, legend = False, palette = color_map)
        pred
        # Plot the confidence intervals
        for cat in df[c].unique():
            #print(cat)
            #print(df.loc[(df[c] == cat), [c]].map(color_map))
            plt.fill_between(df.loc[(df[c] == cat)][x], df.loc[(df[c] == cat)]['conf_int_low'], df.loc[(df[c] == cat)]['conf_int_high'],  color=color_map[cat], alpha=0.3)

        plt.title(f'{f} Fit with {l} link')
        plt.xlabel("Fire Weather Index")
        plt.ylabel("Fire Area Growth km^2")

        #plt.legend(title = pretty_names[c], labels = df.field_response_type.map(pretty_names).unique())
        plt.savefig(f'/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/subsetted_to_{fuel_cat}_model_fit_{f}_with_{l}_{y}_func_of_{x}_by_{c}_bic{model.bic}.png', dpi = 900, transparent = False)
        plt.show()
        
stats_fwi_rolling = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})


print(stats_fwi_rolling[stats_fwi_rolling.Log_Likelyhood.astype("float").min() == stats_fwi_rolling.Log_Likelyhood.astype("float") ])

print(stats_fwi_rolling[stats_fwi_rolling.AIC.astype("float").min() == stats_fwi_rolling.AIC.astype("float") ])
print(stats_fwi_rolling[stats_fwi_rolling.BIC.astype("float").min() == stats_fwi_rolling.BIC.astype("float") ])
stats_fwi_rolling

 # Single Fits

In [ ]:
fuel_cat = "TREES-NE"

df = fire[fire.dominant_landcover == fuel_cat]



df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling"]].dropna()

y = "farea_diff_rolling"
x = "FWI_rolling"
c = "field_response_type"

df = df.sort_values(by = x)

formula = f"{y} ~ {x}"
print(formula)

families = ["NegativeBinomial"] ### Chosen based on BIC (and AIC)
links = ["Log"]
#links = ["Identity"]

ls = []
fs = []
aic = []
ll = []
bic = []
for f in families:
    for l in links: 
        #desc = ModelDesc.from_formula(formula)
        #desc.describe()

        #sm.families.family.Gamma.links
        #link_g = sm.genmod.families.links.Identity()
        #link_g = sm.genmod.families.links.Log()
        #link_g = sm.genmod.families.links.CLogLog()
        #link_g = sm.genmod.families.links.Sqrt()
        #link_g = sm.genmod.families.links.InversePower()
        #link_g = sm.genmod.families.links.NegativeBinomial()
        #link_g = sm.genmod.families.links.Power()
        
        link_g = getattr(sm.genmod.families.links, l)

        method = getattr(sm.families, f)

        model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
        print(model.summary())
        tmp = model.summary2()
        
        fs.append(f)
        ls.append(l)
        aic.append(model.aic)
        ll.append(tmp.tables[0].iloc[2,3])
        bic.append(model.bic)



#         df['fitted'] = model.fittedvalues
#         df['residuals'] = model.resid_response

#         # Plot residuals vs fitted values
#         res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
#         res
#         plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
#         plt.xlabel('Fitted Values')
#         plt.ylabel('Residuals')
#         plt.show()
        

        predictions = model.get_prediction(df, transform = True) #df, transform = False
        df['predicted'] = predictions.predicted_mean
        df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T
        df = df.sort_values(by = x)

        actual = sns.scatterplot(x=x, y=y, data=df)
        actual
        # Plot the fitted values
        pred = sns.lineplot(x=x, y='predicted', data=df, legend = False)
        pred
        # Plot the confidence intervals
        #plt.fill_between(df['FWI_rolling'], df['conf_int_low'], df['conf_int_high'], c ="field_response_type", alpha=0.3)
        
        plt.fill_between(df['FWI_rolling'], df['conf_int_low'], df['conf_int_high'], alpha=0.3)

        plt.title(f'{f} Fit with {l} link')
        plt.xlabel(x)
        plt.ylabel(y)
        #plt.legend()
        plt.savefig(f'/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/single_model_subsetting_by_{fuel_cat}_model_fit_{f}_with_{l}_{y}_func_of_{x}_ONLY_bic{model.bic}.png', dpi = 900, transparent = False)
        plt.show()
        
stats_single_model_fwi_rolling = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})




stats_single_model_fwi_rolling

In [ ]:
fuel_cat = "TREES-NE"

df = fire[fire.dominant_landcover == fuel_cat]

df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling"]].dropna()

y = "farea_diff_rolling"
x = "GEOS5_IMERGEARLY_rolling"
c = "field_response_type"

df = df.sort_values(by = x)

formula = f"{y} ~ {x}"
print(formula)

families = ["NegativeBinomial"] ### Chosen based on BIC (and AIC)
links = ["Log"]
#links = ["Identity"]

ls = []
fs = []
aic = []
ll = []
bic = []
for f in families:
    for l in links: 
        #desc = ModelDesc.from_formula(formula)
        #desc.describe()

        #sm.families.family.Gamma.links
        #link_g = sm.genmod.families.links.Identity()
        #link_g = sm.genmod.families.links.Log()
        #link_g = sm.genmod.families.links.CLogLog()
        #link_g = sm.genmod.families.links.Sqrt()
        #link_g = sm.genmod.families.links.InversePower()
        #link_g = sm.genmod.families.links.NegativeBinomial()
        #link_g = sm.genmod.families.links.Power()
        
        link_g = getattr(sm.genmod.families.links, l)

        method = getattr(sm.families, f)

        model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
        print(model.summary())
        tmp = model.summary2()
        
        fs.append(f)
        ls.append(l)
        aic.append(model.aic)
        ll.append(tmp.tables[0].iloc[2,3])
        bic.append(model.bic)



        df['fitted'] = model.fittedvalues
        df['residuals'] = model.resid_response

        # Plot residuals vs fitted values
        # res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
        # res
        # plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
        # plt.xlabel('Fitted Values')
        # plt.ylabel('Residuals')
        # plt.show()
        

        predictions = model.get_prediction(df, transform = True) #df, transform = False
        df['predicted'] = predictions.predicted_mean
        df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T
        df = df.sort_values(by = x)

        actual = sns.scatterplot(x=x, y=y, data=df)
        actual
        # Plot the fitted values
        pred = sns.lineplot(x=x, y='predicted', data=df, legend = False)
        pred
        # Plot the confidence intervals
        #plt.fill_between(df['FWI_rolling'], df['conf_int_low'], df['conf_int_high'], c ="field_response_type", alpha=0.3)
        
        plt.fill_between(df[x], df['conf_int_low'], df['conf_int_high'], alpha=0.3)

        plt.title(f'{f} Fit with {l} link')
        plt.xlabel(x)
        plt.ylabel(y)
        #plt.legend()
        plt.savefig(f'/projects/old_shared/fire_weather_vis/Lightning_analysis/some_figs/Single_model_subsetting_by_{fuel_cat}_model_fit_{f}_with_{l}_{y}_func_of_{x}_ONLY_bic{model.bic}.png', dpi = 900, transparent = False)
        plt.show()
        
stats_single_model_fwi_rolling = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})




stats_single_model_fwi_rolling

In [ ]:
# model.get_influence

# infl = model.get_influence(observed=False)

# summ_df = infl.summary_frame()
# summ_df.sort_values("cooks_d", ascending=False)[:10]

In [ ]:
# fig = infl.plot_influence()
# fig.tight_layout(pad=1.0)
# fig

In [ ]:
df[df.index == 132]

# Misc exploratory stuff

In [ ]:
## For plot of farea_diff_folling vs GEOS-5.IMERGEEARLY_rolling, two outliers

## Outlier 10972 @ 7/07 -- Still looks like a peak temporal missmatch. Rolling over more numers didn't get rid ove it though. 

### Ourlier 10406 @ 06/22 -- looks like either som small temporal mismatch OR more likely that the IMERGE EARLY FWI got it a little better. 

fire3

In [ ]:
# import geoviews
# import hvplot.pandas
# #from bokeh.models import DatetimeTickFormatter, HoverTool







# fire3[row_mask].hvplot.scatter(x = "GEOS-5.IMERGEARLY_rolling", y = "farea_diff_rolling", hover_cols = ["fireID", "t"] )

    

In [ ]:
#fire3[(fire3.fireID.str.contains("8474")) & (fire3.t == "2023-06-07 12:00:00") ].sort_values(by = "t").fireID.unique()#.explore(style_kwds = {"fillOpacity": 0})


fire3[(fire3.fireID.str.contains("8635"))].sort_values(by = "t").fireID.unique()#.explore(style_kwds = {"fillOpacity": 0})

In [ ]:
fire3[row_mask & (fire3.fireID == "10972") ].sort_values(by = "t") ### Missing some polygons??????????
#print(fire3[row_mask & (fire3.fireID == "10972") & (~fire3.geometry.isna())].t.min())

In [ ]:



density_tansform = fire3[row_mask].groupby(["fireID", "field_response_type"]).farea.max().reset_index()
print(len(density_tansform.fireID.unique()))
print(len(density_tansform.fireID))
#density_tansform

row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)

p = (ggplot(density_tansform, aes(x = "farea", color = "field_response_type"))
        #p = (ggplot(fire3[row_mask], aes( x = x, y = y, color = "farea_shifted"))
     + plotnine.geom_density()
     #+ plotnine.labels.ylab(y)
     + plotnine.labels.xlab("Maximum fire area (km^2) before merging")
     #+ stat_smooth(method = "glm")
     #+ plotnine.scale_y_log10()
     + plotnine.scale_x_log10()
     #+ plotnine.ggtitle(f"{agg_function} in rolling window of {rolling_num} days")

     )
print(p)

In [ ]:
import seaborn as sns

sns.histplot(data=density_tansform, x="farea", hue = "field_response_type")

In [ ]:
p = (ggplot(density_tansform, aes(x = "farea", fill = "field_response_type"))
        #p = (ggplot(fire3[row_mask], aes( x = x, y = y, color = "farea_shifted"))
     + plotnine.geom_histogram(bins = 17)
     #+ plotnine.labels.ylab(y)
     + plotnine.labels.xlab("Maximum fire area (km) before merging")
     #+ stat_smooth(method = "glm")
     #+ plotnine.scale_y_log10()
     + plotnine.scale_x_log10()
     #+ plotnine.ggtitle(f"{agg_function} in rolling window of {rolling_num} days")

     )
print(p)

In [ ]:
density_tansform = fire3[row_mask].groupby(["fireID", "field_response_type"]).farea.min().reset_index()
print(len(density_tansform.fireID.unique()))
print(len(density_tansform.fireID))


p = (ggplot(density_tansform, aes(x = "farea", fill = "field_response_type"))
        #p = (ggplot(fire3[row_mask], aes( x = x, y = y, color = "farea_shifted"))
     + plotnine.geom_histogram(bins = 17)
     #+ plotnine.labels.ylab(y)
     + plotnine.labels.xlab("Fire area (km^2) at start of fire")
     #+ stat_smooth(method = "glm")
     #+ plotnine.scale_y_log10()
     + plotnine.scale_x_log10()
     #+ plotnine.ggtitle(f"{agg_function} in rolling window of {rolling_num} days")

     )
print(p)

In [ ]:

row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)


p = (ggplot(fire3[row_mask], aes(x = "farea_diff", color = "field_response_type"))
        #p = (ggplot(fire3[row_mask], aes( x = x, y = y, color = "farea_shifted"))
     #+ plotnine.geom_histogram(bins = 17)
     + plotnine.geom_density()
     #+ plotnine.labels.ylab(y)
     + plotnine.labels.xlab("difference in fire area (km^2)")
     #+ stat_smooth(method = "glm")
     #+ plotnine.scale_y_log10()
     + plotnine.scale_x_log10()
     #+ plotnine.ggtitle(f"{agg_function} in rolling window of {rolling_num} days")

     )
print(p)

In [ ]:
density_tansform[density_tansform.farea >= 500]

In [ ]:
# #from plotnine import *

# df = pd.DataFrame({
#     'x': [1, 2, 3, 4, 5],
#     'y': [1, 4, 9, 16, 25],
#     'color_var': [1, 1, 2, 2, 3]  # Numerical variable
# })

# # Plot using ggplot (plotnine) with numerical gradient color scale
# p = (ggplot(df, aes(x='x', y='y', color='color_var')) +
#      geom_point() +
#      scale_color_continuous(low="#0000FF", high="#FF0000") +  # Specify the colors for low and high values
#      labs(title='Plot with Numerical Gradient Color Scale')
#     )

# print(p)

In [ ]:
fire3.columns

In [ ]:
# fire3[['fireID', 't', 'farea_diff', 'FWI_diff', 'FWI_rolling',
#        'day_of_fire', 'farea_shifted', 'normalized_farea_diff',
#        'FWI_diff_rolling', 'farea_rolling', 'farea_diff_rolling',
#        'max_dof', 'duration']]

In [ ]:
 p = (ggplot(fire3[row_mask], aes( x = x, y = y, color = "field_response_type"))
         + geom_point()
         + plotnine.labels.ylab(y)
         + plotnine.labels.xlab(x)
         + stat_smooth(method = "glm")
         #+ plotnine.scale_y_log10()
         + plotnine.ggtitle(f"{agg_function} in window of {rolling_num}")

        )
print(p)

In [ ]:
FWI_sorted = fire3[row_mask].sort_values(by = "FWI", ascending= True)

In [ ]:
FWI_sorted[(FWI_sorted.farea_diff >= 50) & FWI_sorted.FWI <= 3].t.astype("datetime64[ns]").hist()

In [ ]:
#FWI_sorted[(FWI_sorted.farea_diff >= 50) & FWI_sorted.FWI <= 3].lat.hist()

In [ ]:
FWI_sorted.columns

# FWI_sorted = FWI_sorted[['fireID', 't', 'geometry',
#        'n_pixels', 'n_newpixels', 'farea', 'fperim', 'flinelen', 'duration',
#        'pixden', 'meanFRP', 'GEOS-5.IMERGEARLY', 'FWI', 'FWI_lead_1',
#        'FWI_lead_2', 'FWI_lead_3', 'FWI_lead_4', 'FWI_lead_5', 'FWI_lead_6',
#        'FWI_lead_7', 'FWI_lead_8', 'farea_diff']]

In [ ]:
big_sp_low_fwi = FWI_sorted[(FWI_sorted.farea_diff >= 50) & FWI_sorted.FWI <= 3]

big_sp_big_fwi = FWI_sorted[(FWI_sorted.farea_diff >= 100) & (FWI_sorted.FWI >= 10)]

low_sp_big_fwi = FWI_sorted[(FWI_sorted.farea_diff <= 100) & (FWI_sorted.FWI >= 10)]

In [ ]:
big_sp_low_fwi = big_sp_low_fwi.to_crs("EPSG:4326")
big_sp_big_fwi = big_sp_big_fwi.to_crs("EPSG:4326")

In [ ]:
big_sp_low_fwi.to_file("/projects/old_shared/fire_weather_vis/Lightning_analysis/Shapefiles/big_spread_low_fwi/low_FWI.shp")

In [ ]:
big_sp_big_fwi.to_file("/projects/old_shared/fire_weather_vis/Lightning_analysis/Shapefiles/big_spread_high_fwi/high_FWI.shp")

In [ ]:
import matplotlib.pyplot as plt

big_sp_big_fwi.t.astype("datetime64[ns]").hist()
plt.xticks(rotation='vertical')

In [ ]:
big_sp_low_fwi.t.astype("datetime64[ns]").hist()

In [ ]:
big_sp_big_fwi

In [ ]:
big_sp_big_fwi

In [ ]:
#import skgstat as skg

In [ ]:
big_sp_big_fwi

In [ ]:
low_sp_big_fwi.columns

In [ ]:
low_sp_big_fwi[['fireID', 't', 'geometry', 'n_pixels', 'n_newpixels', 'farea', 'fperim',
       'flinelen', 'duration', 'meanFRP', 'GEOS-5.IMERGEARLY', 'FWI', 'farea_diff']]

In [ ]:
plt.scatter(low_sp_big_fwi.farea_diff.astype("float"), low_sp_big_fwi["GEOS-5.IMERGEARLY"].astype("float"))
plt.show()

In [ ]:
plt.scatter(low_sp_big_fwi.farea_diff.astype("float"), low_sp_big_fwi["FWI"].astype("float"))
plt.show()

In [ ]:
### Playing with growth rate

fire3["flinelen_shifted"] = fire3.groupby("fireID").flinelen.shift(periods = 1)
fire3["fperim_shifted"] = fire3.groupby("fireID").fperim.shift(periods = 1)
fire3["flinelen_diff"] = fire3.groupby("fireID").flinelen.diff()
fire3["fperim_diff"] = fire3.groupby("fireID").fperim.diff()


### Implementing FBP technical handbook equation 88 https://drive.google.com/file/d/1xBseasON22KFC4jEVBKk75EOuUK-dpM2/view

#1) Calculate the length and breadth of the fires. Simple way, get just the convex hull and the longer one is length. More complex, length shoudl maybe be tied to where active fire line is (to represent head fire?)
from shapely.geometry import LineString

def get_length_to_breadth(polygon):
    
    mbr_points = list(zip(*polygon.minimum_rotated_rectangle.exterior.coords.xy)) ### oh no projection issues? 
    # calculate the length of each side of the minimum bounding rectangle
    mbr_lengths = [LineString((mbr_points[i], mbr_points[i+1])).length for i in range(len(mbr_points) - 1)]

    # get major/minor axis measurements
    minor_axis = min(mbr_lengths)
    major_axis = max(mbr_lengths)
    
    return(major_axis/minor_axis)



fire3["LB"] = fire3.geometry.apply(get_length_to_breadth)


## Calculate spread rate

row_mask = (fire3.fperim_diff > 0.1) & (fire3.flinelen > 0)
perimeter_growth_rate = fire3[row_mask]["fperim_diff"] ### Modify by length of active fire line? 
LB = fire3[row_mask]["LB"]


fire3["spread_rate"] = np.nan
fire3.loc[row_mask, "spread_rate"] = 1/(2*((np.pi * (1+(1/LB)) * (1 + ((LB -1)/(2*(LB+1)))**2)) / perimeter_growth_rate ))


fire3["flinelen_rolling"]  = fire3.groupby("fireID").flinelen.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3["flinelen_diff_rolling"] = fire3.groupby("fireID").flinelen_diff.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3["fperim_rolling"] = fire3.groupby("fireID").fperim.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3["fperim_diff_rolling"] = fire3.groupby("fireID").fperim_diff.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3["spread_rate_rolling"] = fire3.groupby("fireID").spread_rate.rolling(rolling_num).agg(agg_function).reset_index(drop = True)


long_fires = fire3[fire3.day_of_fire > 3].fireID.unique()
#fire3['max_dof'] = fire3.groupby("fireID").day_of_fire.max()

row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)

#x_var = ["FWI","FWI_rolling", "FWI_diff_rolling", "FWI_norm_rolling"]
#y_var = ['farea', "normalized_farea_diff", 'farea_rolling', 'farea_diff_rolling']

#x_var = ["FWI","FWI_rolling", "FWI_diff_rolling"]
#y_var = ["flinelen", "flinelen_rolling", "flinelen_diff_rolling", "fperim_rolling", "fperim_diff_rolling", "spread_rate", "spread_rate_rolling"]

x_var = ["FWI","FWI_rolling", "FWI_diff_rolling", "GEOS-5.IMERGEARLY", "GEOS-5.IMERGEARLY_rolling"]
y_var = ["flinelen_shifted", "spread_rate", "spread_rate_rolling"]


for x in x_var:
    for y in y_var:
        #p = (ggplot(fire3[row_mask], aes( x = x, y = y))
        p = (ggplot(fire3[row_mask], aes( x = x, y = y, color = "field_response_type"))
        #p = (ggplot(fire3[row_mask], aes( x = x, y = y, color = "farea_shifted"))
         + geom_point()
         + plotnine.labels.ylab(y)
         + plotnine.labels.xlab(x)
         + stat_smooth(method = "glm")
         #+ plotnine.scale_y_log10()
         + plotnine.ggtitle(f"{agg_function} in rolling window of {rolling_num} days")

         )
        print(p)
        #del(p)

In [ ]:
row_mask = fire3.farea_diff >0.1

# plt.scatter(fire3[row_mask].farea_diff, ((fire3[row_mask].flinelen_shifted) * fire3[row_mask]["GEOS-5.IMERGEARLY"]))
# plt.show()

y = "farea_diff"



p = (ggplot(fire3[row_mask], aes(x = (fire3[row_mask].flinelen_shifted) * fire3[row_mask]["GEOS-5.IMERGEARLY"], y = y, color = "field_response_type"))
+ geom_point()
+ plotnine.labels.ylab(y)
+ plotnine.labels.xlab("fireline len * FWI")
+ stat_smooth(method = "glm")
+ plotnine.ggtitle(f"{agg_function} in rolling window of {rolling_num} days"))
print(p)

In [ ]:
y = "fperim_diff"


p = (ggplot(fire3[row_mask], aes(x = (fire3[row_mask].flinelen_shifted) * fire3[row_mask]["GEOS-5.IMERGEARLY"], y = y, color = "field_response_type"))
+ geom_point()
+ plotnine.labels.ylab(y)
+ plotnine.labels.xlab("fireline len * FWI")
+ stat_smooth(method = "glm")
+ plotnine.ggtitle(f"{agg_function} in rolling window of {rolling_num} days"))
print(p)

In [ ]:
fuels.to_file('fires_with_dominant_landcover_test_20240722.csv')